In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith

import json
import logging
import re
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from asv.commands.publish import Publish
from asv.config import Config
from asv.util import write_json

from datasmith import setup_environment
from datasmith.docker.context import ContextRegistry, Task
from datasmith.logging_config import configure_logging
from datasmith.scrape.scrape_dashboards import make_benchmark_from_html

configure_logging(level=logging.WARNING)
setup_environment()

/mnt/sdd1/atharvas/formulacode/datasmith


20:56:15 WARNING  simple_useragent.core: Falling back to historic user agent.


In [2]:
context_registry_loc = Path("scratch/merged_context_registry_2025-09-06T06:14:21.165351.json")
benchmark_dir = Path("scratch/artifacts/pipeflush/benchmark_results")
results_dir = benchmark_dir / "results"
dashboard_dir = benchmark_dir / "dashboards"
context_registry = ContextRegistry.load_from_file(context_registry_loc)
assert results_dir.exists(), f"Results dir {results_dir.absolute()} does not exist"
dashboard_dir.mkdir(exist_ok=True, parents=True)

In [10]:
from datasmith.execution.collect_commits import search_commits

merged_sha_commits = search_commits(
    repo_name="pandas-dev/pandas",
    query="state=closed",
    max_pages=4,
    per_page=100,
)

In [12]:
# merged_sha_commits

In [ ]:
# from datasmith.execution.collect_commits_offline import

In [ ]:
from functools import partialmethod

from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)

sha2tasks = {t.sha: t for t in context_registry.registry}


def get_files(pth: Path, pattern: str) -> dict[str, str]:
    return {str(f.relative_to(pth)): f.read_text(encoding="utf-8") for f in pth.rglob(pattern)}


def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def merge_json_dict(dst_path: Path, payload: dict) -> None:
    """Merge dict payload into dst_path (if exists), else write it."""
    if dst_path.exists():
        try:
            existing = json.loads(dst_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            existing = {}
        if not isinstance(existing, dict):
            existing = {}
        existing.update(payload)
        dst_path.write_text(json.dumps(existing, indent=2, sort_keys=True), encoding="utf-8")
    else:
        ensure_dir(dst_path.parent)
        dst_path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def api_publish(repo_root: Path) -> None:
    """
    Publish results using ASV's Python API.
    - Reads repo_root/asv.conf.json
    - Forces results/html dirs to this repo_root
    - Creates Config and runs Publish.run(cfg)
    """
    asv_conf_path = repo_root / "asv.conf.json"
    conf_dict = json.loads(asv_conf_path.read_text(encoding="utf-8"))

    # Ensure minimal keys + local dirs
    conf_dict.setdefault("version", 1)
    conf_dict["results_dir"] = str((repo_root / "results").resolve())
    conf_dict["html_dir"] = str((repo_root / "html").resolve())
    conf_dict["repo_subdir"] = conf_dict.get("repo_subdir", "")
    conf_dict["project"] = str((repo_root / "project").resolve())

    # If repo is a local path, make it absolute; if URL, leave it alone
    repo_val = conf_dict.get("repo")
    if isinstance(repo_val, str) and not re.match(r"^https?://", repo_val):
        conf_dict["repo"] = str(Path(repo_val).resolve())
        print(f"Set repo to local path: {conf_dict['repo']}")

    cfg = Config.from_json(conf_dict)

    write_json(path=asv_conf_path, data=conf_dict, api_version=1)
    Publish.run(cfg)


benchmarked_imgs = list(results_dir.rglob("*pkg"))
dashboards = {}
with tempfile.TemporaryDirectory(prefix="asv-publish-") as tmpdirname:
    tmproot = Path(tmpdirname)
    repo_roots = {}
    repokey2task = {}
    for img in benchmarked_imgs:
        commit, tag = img.name.rsplit("-")[-2:]
        task = sha2tasks.get(commit)
        if not task:
            continue
        if "default" in task.owner:
            continue

        repo_key = f"{task.owner}/{task.repo}"
        repokey2task[repo_key] = Task(owner=task.owner, repo=task.repo, sha=None, tag=task.tag)
        contents = get_files(img, "*.json")
        needed = {"asv.conf.json", "benchmarks.json", "machine.json"}
        if not (needed.issubset({Path(p).name for p in contents}) and len(contents) >= 4):
            continue

        repo_root = repo_roots.setdefault(repo_key, tmproot / repo_key)
        results_out = repo_root / "results"
        html_out = repo_root / "html"
        ensure_dir(results_out)
        ensure_dir(html_out)

        asv_conf = json.loads(contents.pop(next(p for p in contents if p.endswith("asv.conf.json"))))
        benchmarks = json.loads(contents.pop(next(p for p in contents if p.endswith("benchmarks.json"))))
        machine = json.loads(contents.pop(next(p for p in contents if p.endswith("machine.json"))))

        merge_json_dict(results_out / "benchmarks.json", benchmarks)

        machine_name = machine.get("machine") or "imported"
        machine_dir = results_out / machine_name
        ensure_dir(machine_dir)

        # Save machine.json inside machine dir (ASV iterates machine.json files under results/) :contentReference[oaicite:3]{index=3}
        (machine_dir / "machine.json").write_text(json.dumps(machine, indent=2, sort_keys=True), encoding="utf-8")

        seen = set()
        for relpath, text in contents.items():
            if not relpath.endswith(".json"):
                continue
            base = Path(relpath).name
            if base in ("asv.conf.json", "benchmarks.json", "machine.json"):
                continue
            out = machine_dir / base
            if out.name in seen or out.exists():
                i = 1
                while True:
                    cand = machine_dir / f"{i}_{base}"
                    if not cand.exists():
                        out = cand
                        break
                    i += 1
            # write_text(out, text)
            out.write_text(text, encoding="utf-8")
            seen.add(out.name)

        # Patch & save config for this temp repo
        asv_conf = dict(asv_conf) if isinstance(asv_conf, dict) else {}
        asv_conf["results_dir"] = "results"
        asv_conf["html_dir"] = "html"
        # repo_url = parse_repo_url(asv_conf, repo_key)
        asv_conf["repo"] = f"https://github.com/{task.owner}/{task.repo}.git"
        asv_conf["show_commit_url"] = f"https://github.com/{task.owner}/{task.repo}/commit/"
        # asv_conf.setdefault("project", "bencrepos/" + repo_key.replace("-", "/"))
        (repo_root / "asv.conf.json").write_text(json.dumps(asv_conf, indent=2, sort_keys=True), encoding="utf-8")

    failures = []
    dashboards = {}

    def publish_one(repo_key, repo_root):
        task = repokey2task[repo_key]
        try:
            api_publish(repo_root)
            print(f"OK: {repo_root}")
            dashboard_collection = make_benchmark_from_html(
                base_url=f"{repo_root}/html",
                html_dir=f"{repo_root}/html",
                force=False,
            )
            if not dashboard_collection:
                print(f"No dashboard generated for: {task.owner}/{task.repo}")
                return (task, None, None)
            db_path = (dashboard_dir / f"{task.owner}_{task.repo}.fc.pkl").resolve()
            dashboard_collection.save(db_path)
            print(f"Dashboard saved to: {db_path}")
            return (task, dashboard_collection, None)  # noqa: TRY300
        except Exception as e:
            print(f"FAILED ({task.owner}/{task.repo}): {e}")
            return (task, None, e)

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {
            executor.submit(publish_one, repo_key, repo_root): repo_key for repo_key, repo_root in repo_roots.items()
        }
        for future in as_completed(futures):
            task, dashboard_collection, exc = future.result()
            if exc is not None or dashboard_collection is None:
                failures.append(task)
            else:
                dashboards[task] = dashboard_collection

    if failures:
        print("\nSome repos failed to publish:", ", ".join(str(t) for t in failures))

02:17:26 WARNING  root: `wheel_cache_size` has been renamed to `build_cache_size`. Update your `asv.conf.json` accordingly.


OK: /tmp/asv-publish-1xotar5c/numpy/numpy-financial
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/numpy_numpy-financial.fc.pkl
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/numpy_numpy-financial.fc.pkl
OK: /tmp/asv-publish-1xotar5c/DASDAE/dascore
OK: /tmp/asv-publish-1xotar5c/DASDAE/dascore
OK: /tmp/asv-publish-1xotar5c/google-deepmind/mujoco_warp


/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:437: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_benchmarks = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:461: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_summaries_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()


Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/DASDAE_dascore.fc.pkl
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/DASDAE_dascore.fc.pkl
OK: /tmp/asv-publish-1xotar5c/pybamm-team/liionpack
OK: /tmp/asv-publish-1xotar5c/tensorwerk/hangar-py
OK: /tmp/asv-publish-1xotar5c/pybamm-team/liionpack
OK: /tmp/asv-publish-1xotar5c/tensorwerk/hangar-py
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/google-deepmind_mujoco_warp.fc.pkl
OK: /tmp/asv-publish-1xotar5c/glotzerlab/signac


/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:437: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_benchmarks = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:461: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_summaries_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()


...

OK: /tmp/asv-publish-1xotar5c/datalad/datalad
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/kedro-org_kedro.fc.pkl
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/datalad_datalad.fc.pkl
OK: /tmp/asv-publish-1xotar5c/scikit-learn/scikit-learn


/home/asehgal/formulacode/datasmith/src/datasmith/scrape/scrape_dashboards.py:461: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_summaries_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()


Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/scikit-learn_scikit-learn.fc.pkl
OK: /tmp/asv-publish-1xotar5c/pandas-dev/pandas


02:24:12 WARNING  root: Couldn't find 784f4d19 in branches (HEAD)


OK: /tmp/asv-publish-1xotar5c/Unidata/MetPy
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/pandas-dev_pandas.fc.pkl
Dashboard saved to: /home/asehgal/formulacode/datasmith/scratch/artifacts/pipeflush/benchmark_results/dashboards/Unidata_MetPy.fc.pkl

Some repos failed to publish: Task(owner='quantumlib', repo='Cirq', sha=None, commit_date=0.0, tag='pkg')


In [ ]:
# repo_root = Path("quantumlib/Cirq")
# api_publish(repo_root)
# dashboard_collection = make_benchmark_from_html(
#     base_url=f"{repo_root}/html",
#     html_dir=f"{repo_root}/html",
#     force=False,
# )
# some problem with asv. TODO: debug later.


In [ ]:
# collections = [BenchmarkCollection.load(p) for p in Path("scratch/artifacts/processed").rglob("*breakpoints.fc.pkl")]
list(Path("scratch/artifacts/pipeflush").rglob("*breakpoints.fc.pkl"))

[PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/scverse_spatialdata.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/shapely_shapely.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/innobi_pantab.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/pybop-team_PyBOP.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/scipy_scipy.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/mongodb-labs_mongo-arrow.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/GAA-UAM_scikit-fda.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/pyapp-kit_psygnal.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/dwavesystems_dimod.fc.pkl'),
 PosixPath('scratch/artifacts/pipeflush/benchmark_results/dashboards/DASDAE_dascore.fc.pkl'),
 PosixPath('scratch/artifacts/pipef